In [ ]:
from aria_exported import run_discovery

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-c", """
from playwright.sync_api import sync_playwright

def run_discovery(url):
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        try:
            page = browser.new_page()
            page.goto(url, wait_until="domcontentloaded")
            page.wait_for_selector("select", timeout=5000)
            snapshot_yaml = page.locator("body").aria_snapshot()
            print(snapshot_yaml)
            return snapshot_yaml
        except Exception as e:
            print(f"Error: {e}")
            return None
        finally:
            browser.close()

run_discovery("https://www.justdoorsuk.com/window-order.php?product=white-window-style-1")
"""],
    capture_output=True, text=True
)

aria_snapshot = result.stdout

In [ ]:
print(aria_snapshot)

In [ ]:
print(aria_snapshot)

In [ ]:
import re

def find_element_by_label(snapshot: str, label: str):
    """
    Find an element in ARIA snapshot by its label and return xpath candidates.
    """
    lines = snapshot.split('\n')
    results = []
    
    for i, line in enumerate(lines):
        if label.lower() in line.lower():
            # Get indentation level
            indent = len(line) - len(line.lstrip())
            element_info = {
                'line': i,
                'raw': line.strip(),
                'indent': indent,
                'type': None,
                'name': None,
                'xpath': None
            }
            
            # Detect element type and name
            if 'combobox' in line:
                name = re.search(r'combobox "([^"]+)"', line)
                element_info['type'] = 'combobox'
                element_info['name'] = name.group(1) if name else label
                element_info['xpath'] = f'//select[@aria-label="{element_info["name"]}"] | //*[@role="combobox" and @aria-label="{element_info["name"]}"]'
                
            elif 'textbox' in line:
                name = re.search(r'textbox "([^"]+)"', line)
                element_info['type'] = 'textbox'
                element_info['name'] = name.group(1) if name else label
                element_info['xpath'] = f'//input[@aria-label="{element_info["name"]}"] | //input[@placeholder="{element_info["name"]}"]'
                
            elif 'button' in line:
                name = re.search(r'button "([^"]+)"', line)
                element_info['type'] = 'button'
                element_info['name'] = name.group(1) if name else label
                element_info['xpath'] = f'//button[@aria-label="{element_info["name"]}"] | //button[contains(text(), "{element_info["name"]}")]'
                
            elif 'link' in line:
                name = re.search(r'link "([^"]+)"', line)
                element_info['type'] = 'link'
                element_info['name'] = name.group(1) if name else label
                element_info['xpath'] = f'//a[@aria-label="{element_info["name"]}"] | //a[contains(text(), "{element_info["name"]}")]'
                
            elif 'text:' in line:
                element_info['type'] = 'text'
                element_info['name'] = label
                # Look at next line for the actual input element
                if i + 1 < len(lines):
                    next_line = lines[i + 1]
                    if 'combobox' in next_line:
                        name = re.search(r'combobox "([^"]+)"', next_line)
                        n = name.group(1) if name else label
                        element_info['xpath'] = f'//select[@aria-label="{n}"] | //*[@role="combobox" and @aria-label="{n}"]'
                        element_info['type'] = 'combobox (via label)'
                    elif 'textbox' in next_line:
                        name = re.search(r'textbox "([^"]+)"', next_line)
                        n = name.group(1) if name else label
                        element_info['xpath'] = f'//input[@aria-label="{n}"]'
                        element_info['type'] = 'textbox (via label)'
                    else:
                        element_info['xpath'] = f'//*[contains(text(), "{label}")]'
            else:
                element_info['type'] = 'text/other'
                element_info['xpath'] = f'//*[contains(text(), "{label}")]'
            
            results.append(element_info)
    
    return results


def get_xpath(snapshot: str, label: str, verbose=True):
    """
    Simple one-liner: give label, get xpath back.
    """
    results = find_element_by_label(snapshot, label)
    
    if not results:
        print(f"❌ Label '{label}' not found in snapshot.")
        return None
    
    if verbose:
        for r in results:
            print(f"✅ Found: [{r['type']}] → {r['name']}")
            print(f"   XPath: {r['xpath']}\n")
    
    return results[0]['xpath']  # return best match

In [ ]:
xpath = get_xpath(aria_snapshot, "Pipe Outside Diameter mm")
xpath